# Gender Agreement Benchmark v2

Evaluates the impact of ablating Feature 1215 on a **measurable task**: given a context with a feminine or masculine subject, the model must predict the adjective with correct gender agreement.

**Approach:** measure P("bonita") vs P("bonito") after "A menina é muito" — here Feature 1215 is already active due to the preceding feminine context, so ablation should impact agreement.

**Metrics:**
- Baseline vs. ablated accuracy (Feature 1215)
- Accuracy with ablation of 5 random norm-matched features (control)
- Analysis by type (feminine vs. masculine)
- Fisher exact test

**Runtime:** ~10 min on T4

In [ ]:
!pip install transformer_lens sae_lens -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE_DIR = "./data/checkpoints/"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import torch, numpy as np, json, time

device = "cuda" if torch.cuda.is_available() else "cpu"
LAYER = 13
SAE_RELEASE = "gemma-scope-2b-pt-res-canonical"
SAE_ID = f"layer_{LAYER}/width_16k/canonical"
HOOK_NAME = f"blocks.{LAYER}.hook_resid_post"
TARGET_FEATURE = 1215

from sae_lens import SAE, HookedSAETransformer
print("Carregando modelo + SAE...")
model = HookedSAETransformer.from_pretrained_no_processing("gemma-2-2b", device=device, dtype=torch.float16)
tokenizer = model.tokenizer
sae = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
print("Pronto.")

## Benchmark: 50 sentences with adjective gender agreement

Each item has: context with a gendered subject + correct adjective + incorrect adjective.
The context activates Feature 1215 (for feminine items), and the model must predict the agreeing adjective.

In [ ]:
# (context, correct_adjective, incorrect_adjective, gender)
# The context contains subject + copula verb, ending at the point where the adjective should appear.
# For feminine items, Feature 1215 should be active due to the preceding context "A menina é muito".
BENCHMARK = [
    # --- Feminine (correct adjective = feminine) ---
    ("A menina é muito", "bonita", "bonito", "F"),
    ("A professora é muito", "dedicada", "dedicado", "F"),
    ("A diretora é muito", "competente", "competente", "F"),  # invariable, but we keep
    ("A médica foi muito", "elogiada", "elogiado", "F"),
    ("A engenheira é muito", "talentosa", "talentoso", "F"),
    ("A vizinha está muito", "preocupada", "preocupado", "F"),
    ("A casa é muito", "bonita", "bonito", "F"),
    ("A escola ficou muito", "organizada", "organizado", "F"),
    ("A porta está", "aberta", "aberto", "F"),
    ("A menina ficou muito", "feliz", "feliz", "F"),  # invariable
    ("A mãe estava muito", "cansada", "cansado", "F"),
    ("A aluna é muito", "estudiosa", "estudioso", "F"),
    ("A cidade está muito", "movimentada", "movimentado", "F"),
    ("A comida ficou muito", "salgada", "salgado", "F"),
    ("A água está muito", "fria", "frio", "F"),
    ("A música é muito", "animada", "animado", "F"),
    ("A rua estava muito", "escura", "escuro", "F"),
    ("A mesa está muito", "suja", "sujo", "F"),
    ("A roupa ficou muito", "molhada", "molhado", "F"),
    ("A cadeira está muito", "velha", "velho", "F"),
    ("A praia estava muito", "lotada", "lotado", "F"),
    ("A filha dele é muito", "inteligente", "inteligente", "F"),  # invariable
    ("A parede ficou muito", "manchada", "manchado", "F"),
    ("A festa foi muito", "animada", "animado", "F"),
    ("A carta era muito", "longa", "longo", "F"),
    # --- Masculine (correct adjective = masculine) ---
    ("O menino é muito", "bonito", "bonita", "M"),
    ("O professor é muito", "dedicado", "dedicada", "M"),
    ("O diretor é muito", "competente", "competente", "M"),  # invariable
    ("O médico foi muito", "elogiado", "elogiada", "M"),
    ("O engenheiro é muito", "talentoso", "talentosa", "M"),
    ("O vizinho está muito", "preocupado", "preocupada", "M"),
    ("O carro é muito", "bonito", "bonita", "M"),
    ("O museu ficou muito", "organizado", "organizada", "M"),
    ("O portão está", "aberto", "aberta", "M"),
    ("O menino ficou muito", "feliz", "feliz", "M"),  # invariable
    ("O pai estava muito", "cansado", "cansada", "M"),
    ("O aluno é muito", "estudioso", "estudiosa", "M"),
    ("O centro está muito", "movimentado", "movimentada", "M"),
    ("O bolo ficou muito", "salgado", "salgada", "M"),
    ("O suco está muito", "frio", "fria", "M"),
    ("O barulho é muito", "alto", "alta", "M"),
    ("O quarto estava muito", "escuro", "escura", "M"),
    ("O chão está muito", "sujo", "suja", "M"),
    ("O prato ficou muito", "molhado", "molhada", "M"),
    ("O sofá está muito", "velho", "velha", "M"),
    ("O parque estava muito", "lotado", "lotada", "M"),
    ("O filho dele é muito", "inteligente", "inteligente", "M"),  # invariable
    ("O muro ficou muito", "manchado", "manchada", "M"),
    ("O evento foi muito", "animado", "animada", "M"),
    ("O livro era muito", "longo", "longa", "M"),
]

# Remove items with invariable adjectives (correct == incorrect)
BENCHMARK = [(ctx, corr, incorr, g) for ctx, corr, incorr, g in BENCHMARK if corr != incorr]

print(f"Total items (excluding invariable): {len(BENCHMARK)}")
print(f"Femininos: {sum(1 for _,_,_,g in BENCHMARK if g == 'F')}")
print(f"Masculinos: {sum(1 for _,_,_,g in BENCHMARK if g == 'M')}")

In [ ]:
def get_next_token_probs(model, tokenizer, text, hook_name=None, hook_fn=None):
    input_ids = tokenizer.encode(text, return_tensors="pt").to(device)
    with torch.no_grad():
        if hook_fn and hook_name:
            logits = model.run_with_hooks(input_ids, fwd_hooks=[(hook_name, hook_fn)])
        else:
            logits = model(input_ids)
    probs = torch.softmax(logits[0, -1, :].float(), dim=-1)
    return probs


def make_ablation_hook(sae, feature_id, hook_name=HOOK_NAME):
    fid_t = torch.tensor([feature_id], device=device)
    def hook_fn(value, hook):
        with torch.no_grad():
            sae_acts = sae.encode(value)
            modified = sae_acts.clone()
            modified[:, :, fid_t] = 0.0
            reconstructed = sae.decode(modified)
            error = value - sae.decode(sae_acts)
            return reconstructed + error
    return hook_fn


def get_token_prob(probs, tokenizer, word):
    """Sum probabilities over all token IDs that encode this word."""
    ids = tokenizer.encode(" " + word, add_special_tokens=False)
    ids_no_space = tokenizer.encode(word, add_special_tokens=False)
    all_ids = list(set(ids + ids_no_space))
    return sum(probs[tid].item() for tid in all_ids)


def evaluate_benchmark(model, sae, tokenizer, benchmark, feature_id=None):
    results = []
    hook_fn = make_ablation_hook(sae, feature_id) if feature_id is not None else None

    for context, correct_adj, incorrect_adj, gender in benchmark:
        prompt = context + " "
        probs = get_next_token_probs(
            model, tokenizer, prompt,
            hook_name=HOOK_NAME if hook_fn else None,
            hook_fn=hook_fn,
        )

        p_correct = get_token_prob(probs, tokenizer, correct_adj)
        p_incorrect = get_token_prob(probs, tokenizer, incorrect_adj)

        predicted = "correct" if p_correct > p_incorrect else "incorrect"
        margin = p_correct - p_incorrect

        results.append({
            "context": context, "gender": gender,
            "correct_adj": correct_adj, "incorrect_adj": incorrect_adj,
            "p_correct": p_correct, "p_incorrect": p_incorrect,
            "margin": margin, "predicted": predicted,
        })

    accuracy = sum(1 for r in results if r["predicted"] == "correct") / len(results)
    mean_margin = np.mean([r["margin"] for r in results])
    return accuracy, mean_margin, results

## Run benchmark: baseline + Feature 1215 + 5 random controls

In [ ]:
N_CONTROLS = 5
np.random.seed(42)

decoder_norms = sae.W_dec.data.norm(dim=1).cpu()
target_norm = decoder_norms[TARGET_FEATURE].item()
tolerance = 0.20
norm_lo, norm_hi = target_norm * (1 - tolerance), target_norm * (1 + tolerance)

candidates = [fid for fid in range(sae.cfg.d_sae)
              if fid != TARGET_FEATURE and norm_lo <= decoder_norms[fid].item() <= norm_hi]
control_ids = list(np.random.choice(candidates, size=N_CONTROLS, replace=False))

print(f"Feature 1215 norm: {target_norm:.4f}")
print(f"Controles: {control_ids}")
print(f"Normas controles: {[f'{decoder_norms[fid].item():.4f}' for fid in control_ids]}")

# --- Baseline ---
print("\nAvaliando baseline...")
t0 = time.time()
acc_base, margin_base, res_base = evaluate_benchmark(model, sae, tokenizer, BENCHMARK)
print(f"  Baseline accuracy: {acc_base:.1%} | Mean margin: {margin_base:+.4f} | {time.time()-t0:.0f}s")

# --- Feature 1215 Ablation ---
print("\nEvaluating Feature 1215 ablation...")
acc_1215, margin_1215, res_1215 = evaluate_benchmark(model, sae, tokenizer, BENCHMARK, feature_id=TARGET_FEATURE)
print(f"  Ablated 1215 accuracy: {acc_1215:.1%} | Mean margin: {margin_1215:+.4f}")

# --- Controles ---
control_results = {}
for fid in control_ids:
    print(f"\nAvaliando controle Feature {fid}...")
    acc, margin, res = evaluate_benchmark(model, sae, tokenizer, BENCHMARK, feature_id=fid)
    control_results[fid] = {"accuracy": acc, "margin": margin, "results": res}
    print(f"  Ablated {fid} accuracy: {acc:.1%} | Mean margin: {margin:+.4f}")

total_time = time.time() - t0
print(f"\nDone in {total_time:.0f}s")

## Statistical analysis and comparison

In [ ]:
from scipy import stats

# Resumo
control_accs = [v["accuracy"] for v in control_results.values()]
control_margins = [v["margin"] for v in control_results.values()]

n = len(BENCHMARK)
print("="*60)
print(f"RESULTADOS - Gender Agreement Benchmark ({n} itens)")
print("="*60)
print(f"\n{'Condition':<25} {'Accuracy':>10} {'Margin':>10}")
print("-"*50)
print(f"{'Baseline (no ablation)':<25} {acc_base:>9.1%} {margin_base:>+10.4f}")
print(f"{'Feature 1215 ablation':<25} {acc_1215:>9.1%} {margin_1215:>+10.4f}")
print(f"{'Controls (mean±std)':<25} {np.mean(control_accs):>9.1%} {np.mean(control_margins):>+10.4f}")
for fid in control_ids:
    c = control_results[fid]
    print(f"  Controle {fid:<10}      {c['accuracy']:>9.1%} {c['margin']:>+10.4f}")

acc_drop_1215 = acc_base - acc_1215
acc_drop_controls = [acc_base - a for a in control_accs]
print(f"\nDrop de accuracy:")
print(f"  Feature 1215: {acc_drop_1215:+.1%}")
print(f"  Controls (mean): {np.mean(acc_drop_controls):+.1%}")
print(f"  Controls (max): {max(acc_drop_controls):+.1%}")

specificity_ratio = acc_drop_1215 / max(np.mean(acc_drop_controls), 1e-6)
print(f"\nRatio drop_1215 / drop_controles: {specificity_ratio:.1f}x")

# Errors by type (feminine vs masculine)
fem_idx = [i for i, (_, _, _, g) in enumerate(BENCHMARK) if g == "F"]
masc_idx = [i for i, (_, _, _, g) in enumerate(BENCHMARK) if g == "M"]

n_fem = len(fem_idx)
n_masc = len(masc_idx)
fem_correct_base = sum(1 for i in fem_idx if res_base[i]["predicted"] == "correct")
fem_correct_1215 = sum(1 for i in fem_idx if res_1215[i]["predicted"] == "correct")
masc_correct_base = sum(1 for i in masc_idx if res_base[i]["predicted"] == "correct")
masc_correct_1215 = sum(1 for i in masc_idx if res_1215[i]["predicted"] == "correct")

print(f"\nAnalysis by gender:")
print(f"  Femininos: baseline {fem_correct_base}/{n_fem}, ablated {fem_correct_1215}/{n_fem}")
print(f"  Masculinos: baseline {masc_correct_base}/{n_masc}, ablated {masc_correct_1215}/{n_masc}")

fem_drop = fem_correct_base - fem_correct_1215
masc_drop = masc_correct_base - masc_correct_1215
print(f"  Drop femininos: {fem_drop}")
print(f"  Drop masculinos: {masc_drop}")
if fem_drop > masc_drop:
    print("  → Ablation affects feminine MORE (expected for Feature 1215)")

# Permutation test
all_drops = acc_drop_controls + [acc_drop_1215]
rank_1215 = sorted(all_drops, reverse=True).index(acc_drop_1215) + 1
print(f"\nRank de Feature 1215 entre {N_CONTROLS+1} features: {rank_1215}º (maior drop)")

# Fisher exact test: baseline vs 1215
correct_base = int(round(acc_base * n))
correct_1215 = int(round(acc_1215 * n))
table = [[correct_base, n - correct_base], [correct_1215, n - correct_1215]]
odds_ratio, fisher_p = stats.fisher_exact(table)
print(f"\nFisher exact test (baseline vs 1215): OR={odds_ratio:.2f}, p={fisher_p:.4f}")

# Fisher test on feminine items only
table_fem = [[fem_correct_base, n_fem - fem_correct_base],
             [fem_correct_1215, n_fem - fem_correct_1215]]
or_fem, p_fem = stats.fisher_exact(table_fem)
print(f"Fisher (femininos only): OR={or_fem:.2f}, p={p_fem:.4f}")

## Save results

In [ ]:
results_summary = {
    "experiment": "gender_agreement_benchmark_v2",
    "task": "adjective_gender_agreement",
    "n_items": len(BENCHMARK),
    "n_feminine": n_fem,
    "n_masculine": n_masc,
    "n_controls": N_CONTROLS,
    "target_feature": TARGET_FEATURE,
    "target_norm": target_norm,
    "control_features": [int(fid) for fid in control_ids],
    "control_norms": {str(fid): decoder_norms[fid].item() for fid in control_ids},
    "baseline": {"accuracy": acc_base, "mean_margin": float(margin_base)},
    "ablated_1215": {"accuracy": acc_1215, "mean_margin": float(margin_1215)},
    "controls_summary": {
        "mean_accuracy": float(np.mean(control_accs)),
        "std_accuracy": float(np.std(control_accs)),
        "mean_margin": float(np.mean(control_margins)),
        "individual": {str(fid): {"accuracy": v["accuracy"], "margin": float(v["margin"])}
                       for fid, v in control_results.items()},
    },
    "accuracy_drop": {
        "feature_1215": float(acc_drop_1215),
        "controls_mean": float(np.mean(acc_drop_controls)),
        "controls_max": float(max(acc_drop_controls)),
        "specificity_ratio": float(specificity_ratio),
    },
    "by_gender": {
        "feminine": {"baseline": fem_correct_base, "ablated_1215": fem_correct_1215, "total": n_fem},
        "masculine": {"baseline": masc_correct_base, "ablated_1215": masc_correct_1215, "total": n_masc},
    },
    "fisher_exact": {"odds_ratio": float(odds_ratio), "p_value": float(fisher_p)},
    "fisher_feminine_only": {"odds_ratio": float(or_fem), "p_value": float(p_fem)},
    "item_details_1215": [
        {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in r.items()}
        for r in res_1215
    ],
}

out_path = os.path.join(SAVE_DIR, "exp_gender_benchmark_results.json")
with open(out_path, "w") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False, default=str)

local_path = "/content/exp_gender_benchmark_results.json"
with open(local_path, "w") as f:
    json.dump(results_summary, f, indent=2, ensure_ascii=False, default=str)

print(f"Salvo em: {out_path}")
print(f"Salvo em: {local_path}")
print(f"\nResumo final:")
print(f"  Baseline: {acc_base:.1%}")
print(f"  Feature 1215 ablated: {acc_1215:.1%} (drop: {acc_drop_1215:+.1%})")
print(f"  Controls (mean): {np.mean(control_accs):.1%} (drop: {np.mean(acc_drop_controls):+.1%})")
print(f"  Fisher p (todos): {fisher_p:.4f}")
print(f"  Fisher p (femininos): {p_fem:.4f}")
print(f"  Femininos: baseline {fem_correct_base}/{n_fem} → ablated {fem_correct_1215}/{n_fem}")
print(f"  Masculinos: baseline {masc_correct_base}/{n_masc} → ablated {masc_correct_1215}/{n_masc}")